In [ ]:
import pandas as pd
import snowflake.connector
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import warnings
warnings.filterwarnings("ignore")
from tkinter import Tk
from tkinter.filedialog import askopenfilename
import unicodedata
import re
import os
import torch
import math
from transformers import AutoTokenizer
import numpy as np
import sys



IP=input("Enter the IP_Category [SE-Series, S-Standalone, SEA - Season, E-Episodic: ")
Series= ["Series",'Mini Series','Feature','Special','Short','TV Movie','Made For Video']
Standalone=['Special','Feature | Special','Feature | TV Movie','Short','TV Movie','Feature','Made For Video','Feature | Short','Supplemental','']
Seasons=['Season','Supplemental','Non-IP','Term Deal','Ancillary/Derivative','Consumer Products','Live Stage']
Episodic=['Episode','Special','Short','Segment','Music','Publishing','Pilot','Game','']


# Hide the root window
Tk().withdraw()

# Open file picker dialog
file_path = askopenfilename(title="Select Excel file", filetypes=[("Excel files", "*.xlsx")])

# Load CSV into DataFrame
if file_path:
    INPUT_TITLES = pd.read_excel(file_path)
    INPUT_TITLES_0=INPUT_TITLES.copy()
else:
    print("No file selected.")
    sys.exit(0)

# AKA split
def split_aka(title):
    if pd.isna(title):
        return [title]
    
    parts = re.split(r'\s*(?:\s/\s|\bAKA\b)\s*', title, flags=re.IGNORECASE)
    return [p.strip() for p in parts if p.strip()]



if IP in ("S","SE","SEA"):
    # Apply + explode
    INPUT_TITLES["Features"] = INPUT_TITLES["Features"].astype(str)
    INPUT_TITLES["Features"] = INPUT_TITLES["Features"].apply(split_aka)
    INPUT_TITLES = INPUT_TITLES.explode("Features").reset_index(drop=True)
else:
    # Apply + explode
    INPUT_TITLES["SERIES_TITLE"] = INPUT_TITLES["SERIES_TITLE"].astype(str)
    INPUT_TITLES["Episode"] = INPUT_TITLES["Episode"].astype(str)
    INPUT_TITLES["Episode"] = INPUT_TITLES["Episode"].apply(split_aka)
    INPUT_TITLES = INPUT_TITLES.explode("Episode").reset_index(drop=True)

## Title cleaning for snowflake
def normalize_title_0(title):
    if pd.isna(title):
        return ""
    
    # 1️⃣ Normalize Unicode (NFKD separates accents)
    title = unicodedata.normalize('NFKD', str(title))
    
    # 2️⃣ Remove accent marks (convert → ASCII)
    title = title.encode('ascii', 'ignore').decode('utf-8')
    
    # 3️⃣ Lowercase
    title = title.lower()
    
    # 4️⃣ Remove punctuation
    title = re.sub(r"[^\w\s']", '', title)
    
    # 5️⃣ Remove 4-digit year
    title = re.sub(r'\s*\b\d{4}\b$', '', title)

    # 6️⃣ Remove EDITED VERSION and SUBTITLE VERSION
    title = re.sub(r'\b(SUBTITLED VERSION|EDITED VERSION|SUBTITLED|SUBTITLES)\b', '', title, flags=re.IGNORECASE).strip()
    
    # 7️⃣ Remove extra spaces
    title = " ".join(title.split())
    
    return title


if IP in ("S","SE","SEA"):
    INPUT_TITLES["Features"]=INPUT_TITLES["Features"].apply(normalize_title_0)
    INPUT_TITLES["Features"] = INPUT_TITLES["Features"].str.replace("'", "''")

    Input_titles = ",".join([
        f"('{i}', '{j}', {int(k)})" if not pd.isna(k) else f"('{i}', '{j}', {0})"
        for i, j, k in zip(
            INPUT_TITLES["Sno"],
            INPUT_TITLES["Features"],
            INPUT_TITLES["Years"]
        )
    ])
else:
    INPUT_TITLES["Episode"]=INPUT_TITLES["Episode"].apply(normalize_title_0)
    INPUT_TITLES["Episode"]=INPUT_TITLES["Episode"].str.replace("'", "''")
    Input_titles=",".join([f"('{i}', '{j}', {int(k)})" if not pd.isna(k) else f"('{i}', '{j}', {0})" for i,j,k in zip(INPUT_TITLES["Sno"],INPUT_TITLES["Episode"],INPUT_TITLES["Years"])])

## Filtered data from snowflake and check all the Pid's present:

# Create connection
conn = snowflake.connector.connect(
    user='MUVEESHKUMAR.SHANMUGAM@WBD.COM',
    password='MUVEE@23devamanohari', # optional
    account='WBD-COMMONDATAPROD',   
    database='BOLT_MSC_CDS_PROD',
    schema='ATOM_BI',
    role='PUBLIC',
    authenticator='externalbrowser'
)

# Create a cursor
cur = conn.cursor()


# Convert list to SQL-friendly string
if IP=="SE":
    IP_Category = ",".join(f"'{pid}'" for pid in Series)
elif IP=="S":
    IP_Category = ",".join(f"'{pid}'" for pid in Standalone)
elif IP=="E":
    IP_Category = ",".join(f"'{pid}'" for pid in Episodic)
elif IP=="SEA":
    IP_Category = ",".join(f"'{pid}'" for pid in Seasons)

#MPM_Episode_str = ",".join(f"'{pid}'" for pid in Episode)
IS_PARENTS = 'P' if IP in ("SE","SEA") else 'C'

# First query
print("Loading snowflake❄️ data..")
query = fr"""
WITH ATOM_DATA AS (
    SELECT 
        E.SERIES_IDENTIFIER,
        E.NODE_IDENTIFIER,
        E.IP_TYPE,
        E.MPM_NUMBER,
        E.SEASON_MPM_NUMBER,
        E.SERIES_MPM_NUMBER
    FROM BOLT_MSC_CDS_PROD.ATOM_BI.EPISODIC_TITLE_HIERARCHY_VW E
),

HIERARCHY_INFO1 AS (
    SELECT
        E.SERIES_IDENTIFIER,
        E.NODE_IDENTIFIER,
        E.IP_TYPE,
        COALESCE(E.MPM_NUMBER, T.MPM_NUMBER) AS EPISODE_MPM_NUMBER,
        COALESCE(E.SEASON_MPM_NUMBER, T.MPM_PRODUCT_NUMBER) AS SEASON_MPM_NUMBER,
        COALESCE(E.SERIES_MPM_NUMBER, T.MPM_FAMILY_NUMBER) AS SERIES_MPM_NUMBER,
        T.PROPERTY_ID,
        T.PI_UUID
    FROM ATOM_DATA E
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE T
        ON E.NODE_IDENTIFIER = T.NODE_IDENTIFIER
),

SERIES_LOOKUP AS (
    SELECT
        SERIES_MPM_NUMBER,
        MAX(SERIES_IDENTIFIER) AS SERIES_IDENTIFIER
    FROM HIERARCHY_INFO1
    GROUP BY SERIES_MPM_NUMBER
),

FILLING_IDENTIFIERS AS (
    SELECT
        SL.SERIES_MPM_NUMBER,
        T.NODE_IDENTIFIER AS SERIES_IDENTIFIER
    FROM SERIES_LOOKUP SL
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE T
        ON SL.SERIES_MPM_NUMBER = T.MPM_NUMBER
    WHERE T.NODE_IDENTIFIER IS NOT NULL
),

HIERARCHY_INFO_F AS (
    SELECT
        COALESCE(H.SERIES_IDENTIFIER, SL.SERIES_IDENTIFIER, SSL.SERIES_IDENTIFIER) AS SERIES_IDENTIFIER,
        SSL.SEASON_IDENTIFIER,
        H.NODE_IDENTIFIER,
        H.IP_TYPE,
        H.EPISODE_MPM_NUMBER,
        H.SEASON_MPM_NUMBER,
        COALESCE(H.SERIES_MPM_NUMBER, SSL.SERIES_MPM_NUMBER) AS SERIES_MPM_NUMBER,
        H.PROPERTY_ID,
        H.PI_UUID
    FROM HIERARCHY_INFO1 H
    LEFT JOIN FILLING_IDENTIFIERS SL
        ON H.SERIES_MPM_NUMBER = SL.SERIES_MPM_NUMBER
    LEFT JOIN (
        SELECT
            SEASON_IDENTIFIER,
            SEASON_MPM_NUMBER,
            SERIES_IDENTIFIER,
            SERIES_MPM_NUMBER
        FROM (
            SELECT
                H2.SEASON_MPM_NUMBER,
                T.NODE_IDENTIFIER AS SEASON_IDENTIFIER,
                T.MPM_FAMILY_NUMBER AS SERIES_MPM_NUMBER,
                T.NODE_IDENTIFIER AS SERIES_IDENTIFIER,
                ROW_NUMBER() OVER (
                    PARTITION BY H2.SEASON_MPM_NUMBER
                    ORDER BY T.NODE_IDENTIFIER
                ) AS RN
            FROM HIERARCHY_INFO1 H2
            LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE T
                ON H2.SEASON_MPM_NUMBER = T.MPM_NUMBER
        )
        WHERE RN = 1
    ) SSL
        ON H.SEASON_MPM_NUMBER = SSL.SEASON_MPM_NUMBER
),

Heirachy_Integrity1 AS (
    SELECT 
        CASE 
            WHEN HF.NODE_IDENTIFIER = HF.SERIES_IDENTIFIER THEN NULL 
            ELSE HF.SERIES_IDENTIFIER 
        END AS SERIES_IDENTIFIER,
        HF.SEASON_IDENTIFIER,
        HF.NODE_IDENTIFIER,
        HF.IP_TYPE,
        T.MPM_FAMILY_NUMBER,
        T.MPM_PRODUCT_NUMBER,
        T.MPM_NUMBER
    FROM HIERARCHY_INFO_F HF
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE T
        ON HF.NODE_IDENTIFIER = T.NODE_IDENTIFIER
),

Heirachy_Integrity1_series AS (
    SELECT 
        H1.SERIES_IDENTIFIER, H1.SEASON_IDENTIFIER, H1.NODE_IDENTIFIER, H1.IP_TYPE,
        H1.MPM_FAMILY_NUMBER, H1.MPM_PRODUCT_NUMBER, H1.MPM_NUMBER,
        COALESCE(T.LIBRARY_TITLE_FULL, T.LIBRARY_TITLE_SHORT, T.TITLE) AS SERIES_TITLE
    FROM Heirachy_Integrity1 H1
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE T
        ON H1.SERIES_IDENTIFIER = T.NODE_IDENTIFIER
),

Heirachy_Integrity1_season AS (
    SELECT 
        H1.SERIES_IDENTIFIER, H1.SEASON_IDENTIFIER, H1.NODE_IDENTIFIER, H1.IP_TYPE,
        H1.MPM_FAMILY_NUMBER, H1.MPM_PRODUCT_NUMBER, H1.MPM_NUMBER,
        H1.SERIES_TITLE,
        COALESCE(T.LIBRARY_TITLE_FULL, T.LIBRARY_TITLE_SHORT, T.TITLE) AS SEASON_TITLE
    FROM Heirachy_Integrity1_series H1
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE T
        ON H1.SEASON_IDENTIFIER = T.NODE_IDENTIFIER
),

Heirachy_Integrity_episode AS (
    SELECT 
        H1.SERIES_IDENTIFIER, H1.SEASON_IDENTIFIER, H1.NODE_IDENTIFIER, H1.IP_TYPE,
        H1.MPM_FAMILY_NUMBER, H1.MPM_PRODUCT_NUMBER, H1.MPM_NUMBER,
        H1.SERIES_TITLE, H1.SEASON_TITLE,
        COALESCE(T.LIBRARY_TITLE_FULL, T.LIBRARY_TITLE_SHORT, T.TITLE) AS EPISODE_TITLE
    FROM Heirachy_Integrity1_season H1
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE T
        ON H1.NODE_IDENTIFIER = T.NODE_IDENTIFIER
),

Heirachy_Integrity AS (
    SELECT *,
    CASE
        WHEN IP_TYPE IN ('Episode','Special','Short','Segment','Music','Publishing','Pilot','Game','') AND REGEXP_LIKE(UPPER(COALESCE(EPISODE_TITLE,'')),
             '^(S\\d+\\s*)?(EPISODE|EPI|EP|E)\\s*#?\\s*\\d+$|^(SEASON|EPISODE|EPI|EP|E|S)$')
        THEN COALESCE(SEASON_TITLE, SERIES_TITLE) || ' ' || EPISODE_TITLE

        WHEN IP_TYPE IN ('Season','Supplemental','Non-IP','Term Deal','Ancillary/Derivative','Consumer Products','Live Stage') AND REGEXP_LIKE(UPPER(COALESCE(EPISODE_TITLE,'')),
             '^(SEASON|SEA|S)\\s*#?\\s*\\d+$|^(SEASON|EPISODE|EPI|EP|E|S)$')
        THEN COALESCE(SERIES_TITLE, SEASON_TITLE) || ' ' || EPISODE_TITLE

        WHEN REGEXP_LIKE(UPPER(COALESCE(EPISODE_TITLE,'')),
             '^(S\\d+\\s*)?(EPISODE|EPI|EP|E)\\s*#?\\s*\\d+$|^(SEASON|SEA|S)\\s*#?\\s*\\d+$|^(SEASON|EPISODE|EPI|EP|E|S)$')
        THEN COALESCE(SERIES_TITLE, SEASON_TITLE) || ' ' || EPISODE_TITLE

        ELSE EPISODE_TITLE
    END AS Titles_heirachy
    FROM Heirachy_Integrity_episode
),

/* ATOM Search (optimized — single D_TITLE scan): */
raw_inputs AS (
    SELECT 
        column1 AS id,
        column2 AS input_title,
        column3 AS input_year
    FROM VALUES 
        /* put your titles here to search Ex: (1,'Journey Among Women',1977) */
        {Input_titles}
),

input_words AS (
    SELECT DISTINCT
        r.id,
        r.input_title,
        r.input_year,
        TRIM(f.value::string) AS input_word
    FROM raw_inputs r,
    LATERAL FLATTEN(
        INPUT => SPLIT(
            REGEXP_REPLACE(LOWER(r.input_title), '[^a-z0-9à-ÿ ]', ''),
            ' '
        )
    ) f
    WHERE TRIM(f.value::string) <> ''
),

input_word_count AS (
    SELECT 
        id,
        COUNT(*) AS total_input_words
    FROM input_words
    GROUP BY id
),

title_base AS (
    /* Primary: LIBRARY_TITLE_FULL */
    SELECT DISTINCT
        m.NODE_IDENTIFIER,
        COALESCE(m.LIBRARY_TITLE_FULL, m.library_title_short) AS LIBRARY_TITLE,
        m.pi_uuid,
        m.MPM_NUMBER,
        m.ip_type,
        COALESCE(m.ORIGINAL_RELEASE_YEAR, m.PRODUCTION_YEAR) AS YEARS,
        TRIM(f.value::string) AS title_word
    FROM BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE m,
    LATERAL FLATTEN(
        INPUT => SPLIT(
            REGEXP_REPLACE(LOWER(COALESCE(m.LIBRARY_TITLE_FULL, m.library_title_short)), '[^a-z0-9à-ÿ ]', ''),
            ' '
        )
    ) f
    WHERE COALESCE(TRIM(m.IP_TYPE), '') IN ({IP_Category})
      AND TRIM(f.value::string) <> ''

    UNION ALL

    /* Secondary: AKA_PKA_TITLES */
    SELECT DISTINCT
        m.NODE_IDENTIFIER,
        m.AKA_PKA_TITLES AS LIBRARY_TITLE,
        m.pi_uuid,
        m.MPM_NUMBER,
        m.ip_type,
        COALESCE(m.ORIGINAL_RELEASE_YEAR, m.PRODUCTION_YEAR) AS YEARS,
        TRIM(f.value::string) AS title_word
    FROM BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE m,
    LATERAL FLATTEN(
        INPUT => SPLIT(
            REGEXP_REPLACE(LOWER(m.AKA_PKA_TITLES), '[^a-z0-9à-ÿ ]', ''),
            ' '
        )
    ) f
    WHERE COALESCE(TRIM(m.IP_TYPE), '') IN ({IP_Category})
      AND TRIM(f.value::string) <> ''

    UNION ALL

    /* Third: TITLE */
    SELECT DISTINCT
        m.NODE_IDENTIFIER,
        m.TITLE AS LIBRARY_TITLE,
        m.pi_uuid,
        m.MPM_NUMBER,
        m.ip_type,
        COALESCE(m.ORIGINAL_RELEASE_YEAR, m.PRODUCTION_YEAR) AS YEARS,
        TRIM(f.value::string) AS title_word
    FROM BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE m,
    LATERAL FLATTEN(
        INPUT => SPLIT(
            REGEXP_REPLACE(LOWER(m.TITLE), '[^[:alnum:] ]', ''),
            ' '
        )
    ) f
    WHERE COALESCE(TRIM(m.IP_TYPE), '') IN ({IP_Category})
      AND TRIM(f.value::string) <> ''

    UNION ALL

    /* Fourth: Generic titles */
    SELECT DISTINCT
        m.NODE_IDENTIFIER,
        H.Titles_heirachy AS LIBRARY_TITLE,
        m.pi_uuid,
        m.MPM_NUMBER,
        m.ip_type,
        COALESCE(m.ORIGINAL_RELEASE_YEAR, m.PRODUCTION_YEAR) AS YEARS,
        TRIM(f.value::string) AS title_word
    FROM BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE m
    LEFT JOIN Heirachy_Integrity H
        ON m.NODE_IDENTIFIER = H.NODE_IDENTIFIER,
    LATERAL FLATTEN(
        INPUT => SPLIT(
            REGEXP_REPLACE(LOWER(H.Titles_heirachy), '[^a-z0-9à-ÿ ]', ''),
            ' '
        )
    ) f
    WHERE COALESCE(TRIM(m.IP_TYPE), '') IN ({IP_Category})
      AND TRIM(f.value::string) <> ''
),

title_words AS (
    SELECT DISTINCT
        NODE_IDENTIFIER,
        LIBRARY_TITLE,
        PI_UUID,
        MPM_NUMBER,
        IP_TYPE,
        YEARS,
        title_word
    FROM title_base
),

matched_titles AS (
    SELECT 
        i.id,
        t.NODE_IDENTIFIER,
        t.LIBRARY_TITLE,
        t.PI_UUID,
        t.MPM_NUMBER,
        t.IP_TYPE,
        t.YEARS,
        i.input_title,
        i.input_year,
        COUNT(DISTINCT t.title_word) AS matched_words
    FROM input_words i
    JOIN title_words t
        ON i.input_word = t.title_word
    GROUP BY 
        i.id,
        t.NODE_IDENTIFIER,
        t.LIBRARY_TITLE,
        t.PI_UUID,
        t.MPM_NUMBER,
        t.IP_TYPE,
        t.YEARS,
        i.input_title,
        i.input_year
),

Final_matches AS (
    SELECT 
        m.id,
        m.input_title AS Input_title,
        m.input_year,
        m.LIBRARY_TITLE AS Atom_title,
        m.YEARS,
        m.IP_TYPE,
        m.NODE_IDENTIFIER,
        m.MPM_NUMBER,
        m.PI_UUID,
        CASE 
            WHEN m.YEARS IS NULL THEN 'NULL'
            WHEN m.input_year = 0 THEN 'NULL'
            WHEN SPLIT_PART(m.YEARS, '|', 1) BETWEEN m.input_year - 3 AND m.input_year + 3 
                THEN 'GOOD_MATCH'
            ELSE 'CHECK_YEAR'
        END AS YEAR_MATCH_FLAG
    FROM matched_titles m
    JOIN input_word_count iwc
        ON m.id = iwc.id
    WHERE m.matched_words >= CEIL(0.20 * iwc.total_input_words)
),

Atom_title_matching AS (
    SELECT *
    FROM Final_matches
    WHERE YEAR_MATCH_FLAG IN ('NULL', 'GOOD_MATCH')
),

FINAL_OUTPUT AS (
    SELECT 
        A.*,
        CASE
            WHEN EXISTS (
                SELECT 1 FROM Heirachy_Integrity H
                WHERE H.SERIES_IDENTIFIER = A.NODE_IDENTIFIER
            ) THEN 'P'
            ELSE 'C'
        END AS IS_PARENT
    FROM Atom_title_matching A
),

Final_matching_titles AS (
    SELECT
        id,
        input_title,
        Atom_title,
        input_year,
        YEARS,
        F.IP_TYPE,
        F.NODE_IDENTIFIER,
        F.MPM_NUMBER,
        F.PI_UUID,
        H.SERIES_IDENTIFIER
    FROM FINAL_OUTPUT F
    LEFT JOIN Heirachy_Integrity H
        ON F.NODE_IDENTIFIER = H.NODE_IDENTIFIER
    WHERE IS_PARENT = '{IS_PARENTS}'
),

FINAL_OUTPUT_last AS (
    SELECT
        id,
        input_title,
        Atom_title,
        input_year,
        YEARS,
        CASE
          WHEN REGEXP_LIKE(COALESCE(A1.ALTERNATE_IDENTIFIERS_VALUE,''), 'tt[0-9]+', 'i')
          THEN REGEXP_SUBSTR(A1.ALTERNATE_IDENTIFIERS_VALUE, 'tt[^ ]+', 1, 1, 'i')
        END AS TT_CODES,
        F.IP_TYPE,
        F.NODE_IDENTIFIER,
        F.MPM_NUMBER,
        A1.MPM_PRODUCT_NUMBER,
        F.PI_UUID,
        A1.PROPERTY_ID,
        A1.HBO_ID,
        A1.META_ID,
        A1.TURNER_TITLEID,
        A1.MMS3_MCode,
        A1.DASH_Title_ID,
        A1.Aleph_ID,
        A1.iBroadcast_EMEA_ID,
        A1.iBroadcast_APAC_ID,
        A.NODE_IDENTIFIER AS PARENT_ENTITY,
        COALESCE(A.LIBRARY_TITLE_FULL, A.LIBRARY_TITLE_SHORT) AS PARENT_TITLE,
        A.MPM_NUMBER AS PARENT_MPM,
        CASE WHEN (UPPER(A1.LIBRARY_TITLE_FULL) LIKE 'SDS%' OR UPPER(A1.LIBRARY_TITLE_FULL) LIKE '%SDS%' OR UPPER(A1.LIBRARY_TITLE_FULL) LIKE '%DO NOT USE%') OR
                  (UPPER(A1.LIBRARY_TITLE_SHORT) LIKE 'SDS%' OR UPPER(A1.LIBRARY_TITLE_SHORT) LIKE '%SDS%' OR UPPER(A1.LIBRARY_TITLE_SHORT) LIKE '%DO NOT USE%') OR
                  (UPPER(A1.TITLE) LIKE 'SDS%' OR UPPER(A1.TITLE) LIKE '%SDS%' OR UPPER(A1.TITLE) LIKE '%DO NOT USE%') 
             THEN 'Reject'
             ELSE 'Pass' 
        END AS SDS_CHECK_FLAG
    FROM Final_matching_titles F
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE A1
        ON F.NODE_IDENTIFIER = A1.NODE_IDENTIFIER
    LEFT JOIN BOLT_MSC_CDS_PROD.ATOM_BI.D_TITLE A
        ON F.SERIES_IDENTIFIER = A.NODE_IDENTIFIER
),
FINAL_OUTPUT_last as (
SELECT * 
FROM FINAL_OUTPUT_last
WHERE SDS_CHECK_FLAG='Pass'
),
children_summary AS (
    SELECT
        SERIES_IDENTIFIER,
        LISTAGG(IP_TYPE || '[' || cnt || ']', ' + ') WITHIN GROUP (ORDER BY IP_TYPE) AS CHILDREN_COUNT
    FROM (
        SELECT SERIES_IDENTIFIER, IP_TYPE, COUNT(*) AS cnt
        FROM Heirachy_Integrity
        GROUP BY SERIES_IDENTIFIER, IP_TYPE
    )
    GROUP BY SERIES_IDENTIFIER
),
Snowflake_final_result AS (
    SELECT 
        r.id,
        r.input_title,
        F.ATOM_TITLE,
        r.input_year,
        F.YEARS,
        F.TT_CODES,
        F.IP_TYPE,
        F.NODE_IDENTIFIER,
        F.MPM_NUMBER,
        F.MPM_PRODUCT_NUMBER,
        F.PI_UUID,
        F.PROPERTY_ID,
        F.HBO_ID,
        F.META_ID,
        F.TURNER_TITLEID,
        F.MMS3_MCode,
        F.DASH_Title_ID,
        F.Aleph_ID,
        F.iBroadcast_EMEA_ID,
        F.iBroadcast_APAC_ID,
        F.PARENT_ENTITY,
        F.PARENT_TITLE,
        F.PARENT_MPM,
        CS.CHILDREN_COUNT AS CHILDREN_STATUS
    FROM raw_inputs r
    LEFT JOIN FINAL_OUTPUT_last F
        ON r.id = F.id
    LEFT JOIN children_summary CS
        ON F.NODE_IDENTIFIER = CS.SERIES_IDENTIFIER
),

ranked_candidates AS (
    SELECT
        F.*,
        JAROWINKLER_SIMILARITY(LOWER(F.input_title), LOWER(F.ATOM_TITLE)) AS JW,
        EDITDISTANCE(LOWER(F.input_title), LOWER(F.ATOM_TITLE)) AS ED,
        ROW_NUMBER() OVER (
            PARTITION BY F.id
            ORDER BY
                JAROWINKLER_SIMILARITY(LOWER(F.input_title), LOWER(F.ATOM_TITLE)) DESC,
                EDITDISTANCE(LOWER(F.input_title), LOWER(F.ATOM_TITLE)) ASC
        ) AS RN
    FROM Snowflake_final_result F
)

SELECT *,
CASE WHEN NODE_IDENTIFIER IS NULL THEN 'No match' ELSE 'Match' END AS MATCHES
FROM ranked_candidates
WHERE JW >= 50
  AND RN <= 30
ORDER BY id, JW DESC, ED ASC;
"""

with conn.cursor() as cur:
    cur.execute(query)
    rows = cur.fetchall()
    cols = [c[0] for c in cur.description]

SnowFlake_Results = pd.DataFrame(rows, columns=cols)


# Close connection
cur.close()
conn.close()
if IP in ("S","SE","SEA"):
    No_matches=SnowFlake_Results[SnowFlake_Results["MATCHES"]=="No match"]
    Matches=SnowFlake_Results[SnowFlake_Results["MATCHES"]=="Match"]
    Matches["ID"]=Matches["ID"].astype(int)
else:
    No_matches=SnowFlake_Results[SnowFlake_Results["MATCHES"]=="No match"]
    Matches=SnowFlake_Results[SnowFlake_Results["MATCHES"]=="Match"]
    Matches["ID"]=Matches["ID"].astype(int)
    Matches=pd.merge(Matches,INPUT_TITLES,how="left",left_on="ID",right_on="Sno")
    Matches=Matches[["ID","SERIES_TITLE","PARENT_TITLE","INPUT_TITLE","ATOM_TITLE","INPUT_YEAR","TT_CODES","YEARS","IP_TYPE","NODE_IDENTIFIER","MPM_NUMBER","MPM_PRODUCT_NUMBER","PI_UUID","PROPERTY_ID","HBO_ID","META_ID","TURNER_TITLEID",'MMS3_MCODE', 'DASH_TITLE_ID', 'ALEPH_ID', 'IBROADCAST_EMEA_ID','IBROADCAST_APAC_ID',"PARENT_ENTITY","PARENT_MPM"]]
## DEFS:

## SEMANTIC SCORING:

# Steps: Clean the text

def normalize_title(title):
    if pd.isna(title):
        return ""
    
    # 1️⃣ Normalize Unicode (NFKD separates accents)
    title = unicodedata.normalize('NFKD', str(title))
    
    # 2️⃣ Remove accent marks (convert → ASCII)
    title = title.encode('ascii', 'ignore').decode('utf-8')
    
    # 3️⃣ Lowercase
    title = title.lower()
    
    # 4️⃣ Remove punctuation
    title = re.sub(r'[^\w\s]', '', title)
    
    # 5️⃣ Remove 4-digit year
    title = re.sub(r'\s*\b\d{4}\b$', '', title)

    # 6️⃣ Remove EDITED VERSION and SUBTITLE VERSION
    title = re.sub(r'\b(SUBTITLED VERSION|EDITED VERSION)\b', '', title, flags=re.IGNORECASE).strip()
    
    # 7️⃣ Remove extra spaces
    title = " ".join(title.split())
    
    return title

# Step 0: Semantic Matching (AI)
print("Loading the Model🤖...")

os.environ["TRANSFORMERS_NO_TF"] = "1"

model = SentenceTransformer("all-MiniLM-L6-v2")

model_name = "cross-encoder/stsb-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model_max = tokenizer.model_max_length
if model_max is None or model_max > 10000:
    model_max = 512

token_lengths = []
for t1, t2 in zip(Matches["INPUT_TITLE"].tolist(), Matches["ATOM_TITLE"].tolist()):
    enc = tokenizer(
        t1, t2,
        truncation=True,
        max_length=model_max
    )
    token_lengths.append(len(enc["input_ids"]))

p95 = int(np.percentile(token_lengths, 95))
chosen_len = min(p95, model_max)


cross_model = CrossEncoder(model_name, max_length=chosen_len+20)  # +10 for safety margin

# =========================================================
# Step 1: Cross‑Encoder Utility
# =========================================================

def apply_cross_encoder(df, colA, colB, semantic_col, out_col, low=80, high=88):

    df[out_col] = None

    mask = (df[semantic_col] >= low) & (df[semantic_col] < high)

    pairs = list(zip(df.loc[mask, colA], df.loc[mask, colB]))

    if not pairs:
        return df

    scores = cross_model.predict(pairs)
    df.loc[mask, out_col] = [round(s * 100, 2) for s in scores]

    return df

# Step 2: Control Extra Words (CRITICAL)

def extra_word_ratio(input_title, candidate_title):
    t1 = set(input_title.lower().split())
    t2 = set(candidate_title.lower().split())

    extra = len(t2 - t1)
    return extra


def extract_numbers(text):
    return re.findall(r'\d+', text)

def numbers_match(a, b):
    return extract_numbers(a) == extract_numbers(b)
# =========================================================
# Step 4: Final Decision (Single Title)
# =========================================================

def final_decision(input_title, candidate_title, semantic_score, cross_score=None):

    input_title = input_title.strip()
    candidate_title = candidate_title.strip()
    extra_words = extra_word_ratio(input_title, candidate_title)
    
    # 🔴 HARD RULE: numbers must match
    if extract_numbers(input_title) != extract_numbers(candidate_title):
        return "Reject"
    
    if "AKA" in input_title.upper():
        if semantic_score >= 50:
            return "Possible Match"

    if len(input_title.split()) <= 2:
        if extra_words > 2 and semantic_score < 90:
            return "Reject"

    if semantic_score >= 88:
        return "Perfect Match"

    if 70 <= semantic_score < 88:
        if cross_score is not None:
            if cross_score >= 85:
                return "Perfect Match"
            elif cross_score >= 75:
                return "Possible Match"
            else:
                return "Reject"
        return "Possible Match"

    return "Reject"

# =========================================================
# Step 5: Combined Series + Episode Logic
# =========================================================

def combined_match_logic(row):

    s0 = row['SEMANTIC_SCORE_0']
    s1 = row['SEMANTIC_SCORE_1']
    c0 = row.get('CROSS_SCORE_0')
    c1 = row.get('CROSS_SCORE_1')
    e0 = row['EXTRA_WORDS_0']
    e1 = row['EXTRA_WORDS_1']

    input_title = str(row['INPUT_TITLE']).strip()
    candidate_title=str(row['ATOM_TITLE']).strip()
    
    # 🔴 HARD RULE: numbers must match
    if extract_numbers(input_title) != extract_numbers(candidate_title):
        return "Reject"
    
    if "AKA" in input_title.upper():
        if s0 >= 75 and s1 >= 50:
            return "Possible Match"

    if len(input_title.split()) <= 2:
        if e1 > 2 and s1 < 90:
            return "Reject"

    if e0 > 3:
        return "Reject"

    if e1 > 4 and s1 < 85:
        return "Reject"

    if s0 >= 75 and (c0 is None or c0 >= 85) and s1 >= 70:
        return "Perfect Match"

    if 70 <= s1 < 88 and c1 is not None:
        if c1 >= 85:
            return "Perfect Match"
        elif c1 >= 75:
            return "Possible Match"
        else:
            return "Reject"

    if s0 >= 75 and s1 >= 75:
        return "Possible Match"

    return "Reject"

    
## SEMANTIC MATCHING IN ACTION:


PARTIAL_THRESHOLD = 50

# Small helper: Top-N per ID by score using one sort + groupby.head (fast)
def top_n_per_id(df, id_col, score_col, n):
    df = df.sort_values([id_col, score_col], ascending=[True, False])
    return df.groupby(id_col, as_index=False).head(n).reset_index(drop=True)


if IP in ("S", "SE", "SEA"):
    # ----------------------------
    # STANDALONE
    # ----------------------------

    Matches["INPUT_TITLE"] = Matches["INPUT_TITLE"].fillna("")
    Matches["ATOM_TITLE"]  = Matches["ATOM_TITLE"].fillna("")

    Matches["INPUT_TITLE_NORM"] = Matches["INPUT_TITLE"].apply(normalize_title)
    Matches["ATOM_TITLE_NORM"]  = Matches["ATOM_TITLE"].apply(normalize_title)

    # Semantic similarity
    def rowwise_cosine(a, b):
        a = torch.nn.functional.normalize(a, p=2, dim=1)
        b = torch.nn.functional.normalize(b, p=2, dim=1)
        return (a * b).sum(dim=1)

    colA_emb = model.encode(Matches["INPUT_TITLE_NORM"].tolist(), convert_to_tensor=True)
    colB_emb = model.encode(Matches["ATOM_TITLE_NORM"].tolist(),  convert_to_tensor=True)

    pair_scores = rowwise_cosine(colA_emb, colB_emb)
    Matches["SEMANTIC_SCORE"] = (pair_scores.cpu().numpy() * 100)

    # Cross encoder
    Matches = apply_cross_encoder(
        Matches, "INPUT_TITLE_NORM", "ATOM_TITLE_NORM",
        "SEMANTIC_SCORE", "CROSS_SCORE"
    )

    # Final decision
    Matches["MATCH_RESULT"] = Matches.apply(
        lambda r: final_decision(
            r["INPUT_TITLE_NORM"], r["ATOM_TITLE_NORM"],
            r["SEMANTIC_SCORE"], r["CROSS_SCORE"]
        ),
        axis=1
    )

    Matches["SEMANTIC_SCORE"] = Matches["SEMANTIC_SCORE"].round(1)
    Matches["EXTRA_WORDS"] = Matches.apply(
        lambda row: extra_word_ratio(row["INPUT_TITLE_NORM"], row["ATOM_TITLE_NORM"]),
        axis=1
    )

    Matches = Matches.drop(columns=["INPUT_TITLE_NORM", "ATOM_TITLE_NORM"], errors="ignore")

    # Keep scored universe (avoid full .copy() unless you need to mutate independently)
    Matches_scored_all = Matches

    # -------- Top match (Top3, non-reject) --------
    Matches_valid = Matches_scored_all[Matches_scored_all["MATCH_RESULT"] != "Reject"]

    Matches_top3 = top_n_per_id(Matches_valid, "ID", "SEMANTIC_SCORE", 3)

    # Convert MPM_NUMBER safely (fast + stable)
    if "MPM_NUMBER" in Matches_top3.columns:
        Matches_top3["MPM_NUMBER"] = [(i if "/" in str(i) else int(i)) if pd.notna(i) else "" for i in Matches_top3["MPM_NUMBER"]]

    # Dedupe after shrinking
    if "NODE_IDENTIFIER" in Matches_top3.columns:
        Matches_top3 = Matches_top3.drop_duplicates(subset=["ID", "NODE_IDENTIFIER"], keep="first")

    # Add INPUT title/features using map (faster than merge)
    features_map = INPUT_TITLES_0.set_index("Sno")["Features"]
    Matches_top3["INPUT_TITLES"] = Matches_top3["ID"].map(features_map)

    # Select final columns (only existing)
    if IP == "SE":
        desired_cols_top = [
            "ID", "INPUT_TITLES", "ATOM_TITLE", "INPUT_YEAR", "YEARS", "TT_CODES", "IP_TYPE",
            "NODE_IDENTIFIER", "MPM_NUMBER", "SEMANTIC_SCORE", "MATCH_RESULT","CHILDREN_STATUS",
            "PI_UUID","HBO_ID","META_ID","TURNER_TITLEID",
            'MMS3_MCODE', 'DASH_TITLE_ID', 'ALEPH_ID', 'IBROADCAST_EMEA_ID','IBROADCAST_APAC_ID'
        ]
    else:
        desired_cols_top = [
            "ID", "INPUT_TITLES", "ATOM_TITLE", "INPUT_YEAR", "YEARS", "TT_CODES", "IP_TYPE",
            "NODE_IDENTIFIER", "MPM_NUMBER", "PARENT_TITLE","PARENT_ENTITY","PARENT_MPM",
            "SEMANTIC_SCORE", "MATCH_RESULT","PI_UUID","PROPERTY_ID","HBO_ID","META_ID","TURNER_TITLEID",
            'MMS3_MCODE', 'DASH_TITLE_ID', 'ALEPH_ID', 'IBROADCAST_EMEA_ID','IBROADCAST_APAC_ID'
        ]
    Matches_top3 = Matches_top3[[c for c in desired_cols_top if c in Matches_top3.columns]]

    top_ids = set(Matches_top3["ID"].dropna().unique())

    # -------- Partial match (Top4 per ID, score>=threshold, excluding top IDs) --------
    Partial_pool = Matches_scored_all[
        (Matches_scored_all["SEMANTIC_SCORE"] >= PARTIAL_THRESHOLD) &
        (~Matches_scored_all["ID"].isin(top_ids))
    ]

    # If you want to exclude rejects in partial too, uncomment:
    # Partial_pool = Partial_pool[Partial_pool["MATCH_RESULT"] != "Reject"]

    Partial_matches = top_n_per_id(Partial_pool, "ID", "SEMANTIC_SCORE", 4)

    if "MPM_NUMBER" in Partial_matches.columns:
        Partial_matches["MPM_NUMBER"] = [(i if "/" in str(i) else int(i)) if pd.notna(i) else "" for i in Partial_matches["MPM_NUMBER"]]

    if "NODE_IDENTIFIER" in Partial_matches.columns:
        Partial_matches = Partial_matches.drop_duplicates(subset=["ID", "NODE_IDENTIFIER"], keep="first")

    Partial_matches["INPUT_TITLES"] = Partial_matches["ID"].map(features_map)
    if IP == "SE":
        desired_cols_partial = [
            "ID", "INPUT_TITLES", "ATOM_TITLE", "INPUT_YEAR", "YEARS", "TT_CODES", "IP_TYPE",
            "NODE_IDENTIFIER", "MPM_NUMBER", "SEMANTIC_SCORE", "MATCH_RESULT","CHILDREN_STATUS",
            "PI_UUID","HBO_ID","META_ID","TURNER_TITLEID",
            'MMS3_MCODE', 'DASH_TITLE_ID', 'ALEPH_ID', 'IBROADCAST_EMEA_ID','IBROADCAST_APAC_ID'
        ]
    else:
        desired_cols_partial = [
            "ID", "INPUT_TITLES", "ATOM_TITLE", "INPUT_YEAR", "YEARS", "TT_CODES", "IP_TYPE",
            "NODE_IDENTIFIER", "MPM_NUMBER", "PARENT_TITLE","PARENT_ENTITY","PARENT_MPM",
            "SEMANTIC_SCORE", "MATCH_RESULT","PI_UUID","PROPERTY_ID","HBO_ID","META_ID","TURNER_TITLEID",
            'MMS3_MCODE', 'DASH_TITLE_ID', 'ALEPH_ID', 'IBROADCAST_EMEA_ID','IBROADCAST_APAC_ID'
            
        ]
    Partial_matches = Partial_matches[[c for c in desired_cols_partial if c in Partial_matches.columns]]
    perfects=Partial_matches[(Partial_matches["MATCH_RESULT"]=="Perfect Match")|(Partial_matches["MATCH_RESULT"]=="Possible Match")|(Partial_matches["SEMANTIC_SCORE"]>=85)]
    perfects["MATCH_RESULT"]=["Possible Match" if i=="Reject" else i for i in perfects["MATCH_RESULT"]]
    Matches_top3=pd.concat([Matches_top3,perfects])
    top_ids = set(Matches_top3["ID"].dropna().unique())
    Partial_matches = Partial_matches[~Partial_matches["ID"].isin(perfects["ID"].unique())]
    partial_ids = set(Partial_matches["ID"].dropna().unique())

    # -------- No match --------
    all_input_ids = set(INPUT_TITLES_0["Sno"].dropna().unique())
    no_match_ids = sorted(INPUT_TITLES_0[~INPUT_TITLES_0["Sno"].isin(top_ids.union(partial_ids))]["Sno"])

    No_match_sheet = (
        INPUT_TITLES_0[INPUT_TITLES_0["Sno"].isin(no_match_ids)][["Sno", "Features","Years"]]
        .rename(columns={"Sno": "ID", "Features": "INPUT_TITLE"})
        .sort_values("ID")
        .reset_index(drop=True)
    )


else:
    # ----------------------------
    # EPISODIC
    # ----------------------------

    for c in ["PARENT_TITLE", "SERIES_TITLE", "INPUT_TITLE", "ATOM_TITLE"]:
        Matches[c] = Matches.get(c, "").fillna("") if c in Matches.columns else ""

    Matches["SERIES_TITLE_NORM"] = Matches["SERIES_TITLE"].apply(normalize_title)
    Matches["PARENT_TITLE_NORM"] = Matches["PARENT_TITLE"].apply(normalize_title)
    Matches["INPUT_TITLE_NORM"]  = Matches["INPUT_TITLE"].apply(normalize_title)
    Matches["ATOM_TITLE_NORM"]   = Matches["ATOM_TITLE"].apply(normalize_title)

    # Semantic scores
    def rowwise_cosine(a, b):
        a = torch.nn.functional.normalize(a, p=2, dim=1)
        b = torch.nn.functional.normalize(b, p=2, dim=1)
        return (a * b).sum(dim=1)
    colA_emb = model.encode(Matches["SERIES_TITLE_NORM"].tolist(), convert_to_tensor=True)
    colB_emb = model.encode(Matches["PARENT_TITLE_NORM"].tolist(), convert_to_tensor=True)
    pair_scores = rowwise_cosine(colA_emb, colB_emb)
    Matches["SEMANTIC_SCORE_0"] = (pair_scores.cpu().numpy() * 100)

    colA_emb = model.encode(Matches["INPUT_TITLE_NORM"].tolist(), convert_to_tensor=True)
    colB_emb = model.encode(Matches["ATOM_TITLE_NORM"].tolist(), convert_to_tensor=True)
    pair_scores = rowwise_cosine(colA_emb, colB_emb)
    Matches["SEMANTIC_SCORE_1"] = (pair_scores.cpu().numpy() * 100)

    # Cross encoders
    Matches = apply_cross_encoder(
        Matches, "SERIES_TITLE_NORM", "PARENT_TITLE_NORM",
        "SEMANTIC_SCORE_0", "CROSS_SCORE_0", 75, 88
    )
    Matches = apply_cross_encoder(
        Matches, "INPUT_TITLE_NORM", "ATOM_TITLE_NORM",
        "SEMANTIC_SCORE_1", "CROSS_SCORE_1", 75, 88
    )

    Matches["MATCH_RESULT_0"] = Matches.apply(
        lambda row: final_decision(row["SERIES_TITLE_NORM"], row["PARENT_TITLE_NORM"], row["SEMANTIC_SCORE_0"]),
        axis=1
    )
    Matches["MATCH_RESULT_1"] = Matches.apply(
        lambda row: final_decision(row["INPUT_TITLE_NORM"], row["ATOM_TITLE_NORM"], row["SEMANTIC_SCORE_1"]),
        axis=1
    )

    Matches["EXTRA_WORDS_0"] = Matches.apply(
        lambda row: extra_word_ratio(row["SERIES_TITLE_NORM"], row["PARENT_TITLE_NORM"]),
        axis=1
    )
    Matches["EXTRA_WORDS_1"] = Matches.apply(
        lambda row: extra_word_ratio(row["INPUT_TITLE_NORM"], row["ATOM_TITLE_NORM"]),
        axis=1
    )

    Matches["FINAL_MATCH_RESULT"] = Matches.apply(combined_match_logic, axis=1)
    Matches["SEMANTIC_SCORE_1"] = Matches["SEMANTIC_SCORE_1"].round(1)

    Matches = Matches.drop(
        columns=["INPUT_TITLE_NORM", "ATOM_TITLE_NORM", "PARENT_TITLE_NORM", "SERIES_TITLE_NORM"],
        errors="ignore"
    )

    Matches_scored_all = Matches

    # -------- Top match (Top3, non-reject) --------
    Matches_valid = Matches_scored_all[Matches_scored_all["FINAL_MATCH_RESULT"] != "Reject"]
    Matches_top3 = top_n_per_id(Matches_valid, "ID", "SEMANTIC_SCORE_1", 3)

    Matches_top3 = Matches_top3.drop(
        columns=["MATCH_RESULT_0", "MATCH_RESULT_1", "SEMANTIC_SCORE_0", "EXTRA_WORDS_0"],
        errors="ignore"
    )
    
    # Add INPUT title/features using map (faster than merge)
    features_map = INPUT_TITLES_0.set_index("Sno")["Episode"]
    Matches_top3["INPUT_TITLE"] = Matches_top3["ID"].map(features_map)
    Matches_top3 = Matches_top3[['ID', 'SERIES_TITLE', 'PARENT_TITLE', 'INPUT_TITLE', 'ATOM_TITLE',
       'INPUT_YEAR', 'YEARS', 'TT_CODES','IP_TYPE', 'NODE_IDENTIFIER',
       'MPM_NUMBER',"PARENT_ENTITY","PARENT_MPM",'SEMANTIC_SCORE_1', 'FINAL_MATCH_RESULT','PI_UUID',"PROPERTY_ID","HBO_ID","META_ID","TURNER_TITLEID",
       'MMS3_MCODE', 'DASH_TITLE_ID', 'ALEPH_ID', 'IBROADCAST_EMEA_ID','IBROADCAST_APAC_ID']]
    
    if "MPM_NUMBER" in Matches_top3.columns:
        Matches_top3["MPM_NUMBER"] = [(i if "/" in str(i) else int(i)) if pd.notna(i) else "" for i in Matches_top3["MPM_NUMBER"]]

    if "NODE_IDENTIFIER" in Matches_top3.columns:
        Matches_top3 = Matches_top3.drop_duplicates(subset=["ID", "NODE_IDENTIFIER"], keep="first")

    top_ids = set(Matches_top3["ID"].dropna().unique())

    # -------- Partial match (Top4 per ID, score>=threshold, excluding top IDs) --------
    Partial_pool = Matches_scored_all[
        (Matches_scored_all["SEMANTIC_SCORE_1"] >= PARTIAL_THRESHOLD) &
        (~Matches_scored_all["ID"].isin(top_ids))
    ]

    # If you want to exclude rejects in partial too, uncomment:
    # Partial_pool = Partial_pool[Partial_pool["FINAL_MATCH_RESULT"] != "Reject"]

    Partial_matches = top_n_per_id(Partial_pool, "ID", "SEMANTIC_SCORE_1", 4)

    Partial_matches = Partial_matches.drop(
        columns=["MATCH_RESULT_0", "MATCH_RESULT_1", "SEMANTIC_SCORE_0", "EXTRA_WORDS_0"],
        errors="ignore"
    )

    # Add INPUT title/features using map (faster than merge)
    features_map = INPUT_TITLES_0.set_index("Sno")["Episode"]
    Partial_matches["INPUT_TITLE"] = Partial_matches["ID"].map(features_map)
    Partial_matches = Partial_matches[['ID', 'SERIES_TITLE', 'PARENT_TITLE', 'INPUT_TITLE', 'ATOM_TITLE',
       'INPUT_YEAR', 'YEARS', 'TT_CODES','IP_TYPE', 'NODE_IDENTIFIER',
       'MPM_NUMBER', "PARENT_ENTITY","PARENT_MPM", 'SEMANTIC_SCORE_1', 'FINAL_MATCH_RESULT',
       'PI_UUID',"PROPERTY_ID","HBO_ID","META_ID","TURNER_TITLEID",
       'MMS3_MCODE', 'DASH_TITLE_ID', 'ALEPH_ID', 'IBROADCAST_EMEA_ID','IBROADCAST_APAC_ID']]
    
    if "MPM_NUMBER" in Partial_matches.columns:
        Partial_matches["MPM_NUMBER"] = [(i if "/" in str(i) else int(i)) if pd.notna(i) else "" for i in Partial_matches["MPM_NUMBER"]]

    if "NODE_IDENTIFIER" in Partial_matches.columns:
        Partial_matches = Partial_matches.drop_duplicates(subset=["ID", "NODE_IDENTIFIER"], keep="first")

    perfects=Partial_matches[(Partial_matches["FINAL_MATCH_RESULT"]=="Perfect Match")|(Partial_matches["FINAL_MATCH_RESULT"]=="Possible Match") & (Partial_matches["SEMANTIC_SCORE_1"]>=85)]
    perfects["FINAL_MATCH_RESULT"]=["Possible Match" if i=="Reject" else i for i in perfects["FINAL_MATCH_RESULT"]]
    Matches_top3=pd.concat([Matches_top3,perfects])
    top_ids = set(Matches_top3["ID"].dropna().unique())
    Partial_matches = Partial_matches[~Partial_matches["ID"].isin(perfects["ID"].unique())]

    partial_ids = set(Partial_matches["ID"].dropna().unique())

    # -------- No match --------
    all_input_ids = set(INPUT_TITLES_0["Sno"].dropna().unique())
    no_match_ids = sorted(INPUT_TITLES_0[~INPUT_TITLES_0["Sno"].isin(top_ids.union(partial_ids))]["Sno"])

    No_match_sheet = INPUT_TITLES_0[INPUT_TITLES_0["Sno"].isin(no_match_ids)].copy()
    keep_cols = [c for c in ["Sno", "SERIES_TITLE", "Episode"] if c in No_match_sheet.columns]
    No_match_sheet = No_match_sheet[keep_cols].rename(columns={"Sno": "ID", "Episode": "INPUT_TITLE"})

    if "INPUT_TITLE" not in No_match_sheet.columns and "Features" in INPUT_TITLES_0.columns:
        No_match_sheet = (
            INPUT_TITLES_0[INPUT_TITLES_0["Sno"].isin(no_match_ids)][["Sno", "Features","Years"]]
            .rename(columns={"Sno": "ID", "Features": "INPUT_TITLE"})
        )

    No_match_sheet = No_match_sheet.sort_values("ID").reset_index(drop=True)

# -----------------------------
# 1) Define column grouping rules (your requirement)
# -----------------------------
GROUPS = {
    "STANDALONE": [
        ("MATCHED OUTPUT", ["ID","INPUT_TITLES","ATOM_TITLE","INPUT_YEAR","YEARS","TT_CODES"]),
        ("STANDALONE INFO", ["IP_TYPE","NODE_IDENTIFIER"]),
        ("PARENT INFO", ["PARENT_TITLE","PARENT_ENTITY","PARENT_MPM"]),
        ("SCORES", ["SEMANTIC_SCORE","MATCH_RESULT"]),
        ("IDENTIFIERS", ["MPM_NUMBER","PI_UUID","PROPERTY_ID","HBO_ID","META_ID","TURNER_TITLEID","MMS3_MCODE",
                         "DASH_TITLE_ID","ALEPH_ID","IBROADCAST_EMEA_ID","IBROADCAST_APAC_ID",
                         "For Ingestion"])
    ],

    "SERIES": [
        ("MATCHED OUTPUT", ["ID","INPUT_TITLES","ATOM_TITLE","INPUT_YEAR","YEARS","TT_CODES"]),
        ("SERIES INFO", ["IP_TYPE","NODE_IDENTIFIER","CHILDREN_STATUS"]),
        ("SCORES", ["SEMANTIC_SCORE","MATCH_RESULT"]),
        ("IDENTIFIERS", ["MPM_NUMBER","PI_UUID","HBO_ID","META_ID","TURNER_TITLEID","MMS3_MCODE",
                         "DASH_TITLE_ID","ALEPH_ID","IBROADCAST_EMEA_ID","IBROADCAST_APAC_ID",
                         "For Ingestion"])
    ],

    "SEASON": [
        ("MATCHED OUTPUT", ["ID","INPUT_TITLES","ATOM_TITLE","INPUT_YEAR","YEARS","TT_CODES"]),
        ("SEASON INFO", ["IP_TYPE","NODE_IDENTIFIER","CHILDREN_STATUS"]),
        ("PARENT INFO", ["PARENT_TITLE","PARENT_ENTITY","PARENT_MPM"]),
        ("SCORES", ["SEMANTIC_SCORE","MATCH_RESULT"]),
        ("IDENTIFIERS", ["MPM_NUMBER","PI_UUID","PROPERTY_ID","HBO_ID","META_ID","TURNER_TITLEID","MMS3_MCODE",
                         "DASH_TITLE_ID","ALEPH_ID","IBROADCAST_EMEA_ID","IBROADCAST_APAC_ID",
                         "For Ingestion"])
    ],

    "EPISODICS": [
        ("MATCHED OUTPUT", ["ID","SERIES_TITLE","PARENT_TITLE","INPUT_TITLE","ATOM_TITLE","INPUT_YEAR","YEARS","TT_CODES"]),
        ("EPISODE INFO", ["IP_TYPE","NODE_IDENTIFIER"]),
        ("PARENT INFO", ["PARENT_ENTITY","PARENT_MPM"]),
        ("SCORES", ["SEMANTIC_SCORE_1","FINAL_MATCH_RESULT"]),
        ("IDENTIFIERS", ["MPM_NUMBER","PI_UUID","PROPERTY_ID","HBO_ID","META_ID","TURNER_TITLEID","MMS3_MCODE",
                         "DASH_TITLE_ID","ALEPH_ID","IBROADCAST_EMEA_ID","IBROADCAST_APAC_ID",
                         "For Ingestion"])
    ]
}

def flatten_groups(groups):
    ordered = []
    for _, cols in groups:
        ordered.extend(cols)
    return ordered


# ✅ Priority columns + EXACT output labels you want
INGESTION_PRIORITY_LABELS = [
    ("MPM_NUMBER",          "MPM_Number"),
    ("PI_UUID",             "PI_UUID"),
    ("PROPERTY_ID",         "Property_ID"),
    ("HBO_ID",              "HBO_ID"),
    ("META_ID",             "Meta_ID"),
    ("TURNER_TITLEID",      "Turner_TitleID"),
    ("MMS3_MCODE",          "MMS3_MCode"),
    ("DASH_TITLE_ID",       "DASH_Title_ID"),
    ("ALEPH_ID",            "Aleph_ID"),
    ("IBROADCAST_EMEA_ID",  "iBroadcast_EMEA_ID"),
    ("IBROADCAST_APAC_ID",  "iBroadcast_APAC_ID"),
]

# ✅ Columns to normalize to numeric where possible:
# - int if integer-like
# - float if decimal
# - else keep original
NUM_COERCE_COLS = [
    "PROPERTY_ID",
    "HBO_ID",
    "META_ID",
    "TURNER_TITLEID",
    "MMS3_MCODE",
    "DASH_TITLE_ID",
    "ALEPH_ID",
    "IBROADCAST_EMEA_ID",
    "IBROADCAST_APAC_ID",
    "PARENT_MPM",
    "YEARS",
]

def try_number_preserve_decimals(x):
    """
    Convert only when safely numeric:
      - integer-like -> int
      - decimal -> float (preserve decimals)
    Otherwise keep value as-is.
    Empty -> NA (writes blank).
    """
    if x is None or pd.isna(x):
        return pd.NA

    # ints
    if isinstance(x, (int, np.integer)):
        return int(x)

    # floats
    if isinstance(x, (float, np.floating)):
        if math.isnan(x) or math.isinf(x):
            return pd.NA
        return int(x) if float(x).is_integer() else float(x)

    s = str(x).strip()
    if s == "":
        return pd.NA

    s_clean = s.replace(",", "")

    # pure integer string
    if re.fullmatch(r"[+-]?\d+", s_clean):
        try:
            return int(s_clean)
        except:
            return x

    # decimal string -> float
    if re.fullmatch(r"[+-]?\d+\.\d+", s_clean):
        try:
            f = float(s_clean)
            return int(f) if float(f).is_integer() else f
        except:
            return x

    return x

def convert_columns_try_number_preserve_decimals(df, cols):
    for col in cols:
        if col in df.columns:
            df[col] = df[col].apply(try_number_preserve_decimals)
    return df


# -----------------------------
# Export
# -----------------------------
with pd.ExcelWriter("Output.xlsx", engine="xlsxwriter") as writer:
    workbook = writer.book

    # Base URLs
    RELTIO_BASE_URL = "https://361.reltio.com/nui/RohAASgkA5WQGA9/profile?entityUri=entities%2F"
    IMDB_BASE_URL   = "https://www.imdb.com/title/"

    # Formats
    header_format = workbook.add_format({
        "bold": True, "text_wrap": True, "valign": "middle", "align": "center",
        "border": 1, "bg_color": "#97F8A9"
    })
    border_format = workbook.add_format({"border": 1})

    id_green  = workbook.add_format({"bg_color": "#3BF160", "border": 1, "bold": True})
    id_yellow = workbook.add_format({"bg_color": "#FCFF34", "border": 1, "bold": True})

    group_band_format = workbook.add_format({
        "bold": True, "align": "center", "valign": "vcenter",
        "border": 1, "bg_color": "#279516", "font_color": "#FFFFFF"
    })

    header_format_match = workbook.add_format({
        "bold": True,
        "text_wrap": False,
        "valign": "middle",
        "align": "center",
        "border": 1,
        "bg_color": "#97F8A9"
    })

    link_format = workbook.add_format({
        "border": 1,
        "font_color": "#0563C1",
        "underline": 1
    })

    # ✅ red format for NO ID/NO_ID in For Ingestion
    no_id_red_format = workbook.add_format({
        "border": 1,
        "font_color": "#FF0000",
        "bold": True
    })

    # ✅ NEW: red highlight for any IDENTIFIERS value containing "|"
    pipe_red_format = workbook.add_format({
        "border": 1,
        "font_color": "#9C0006",
        "bg_color": "#FFC7CE",   # light red fill
        "bold": True
    })

    FIXED_WIDTHS = {
        "MPM_NUMBER": 16,
        "PARENT_MPM": 16,
        "MMS3_MCODE": 16,
        "NODE_IDENTIFIER": 28,
        "TT_CODES": 16,
        "PROPERTY_ID": 14,
        "TURNER_TITLEID": 14,
        "For Ingestion": 18,
    }
    MIN_COL_WIDTH = 10
    MAX_COL_WIDTH = 60


    def apply_grouped_formatting(df, sheet_name, ip_mode):
        groups = GROUPS.get(ip_mode, GROUPS["STANDALONE"])
        ordered_cols = flatten_groups(groups)

        existing_order = [c for c in ordered_cols if c in df.columns]
        leftovers = [c for c in df.columns if c not in existing_order]
        final_cols = existing_order + leftovers
        df2 = df.reindex(columns=final_cols)

        # numeric conversion (int if integer-like else float if decimal else keep)
        df2 = convert_columns_try_number_preserve_decimals(df2, NUM_COERCE_COLS)

        # ✅ IDENTIFIERS columns list for this mode (used for pipe highlighting)
        identifiers_cols = []
        for gname, cols in groups:
            if gname == "IDENTIFIERS":
                identifiers_cols = cols
                break
        identifier_cols_set = set([c for c in identifiers_cols if c in df2.columns])

        # For Ingestion: identifier NAME
        # Ignore values containing "|" for ingestion decision
        conds = []
        labels = []

        for col, out_label in INGESTION_PRIORITY_LABELS:
            if col in df2.columns:
                s_str = df2[col].astype("string").str.strip()
                cond = (
                    s_str.notna()
                    & (s_str != "")
                    & (~s_str.str.contains(r"\|", na=False))
                )
                conds.append(cond.to_numpy())
                labels.append(out_label)

        df2["For Ingestion"] = np.select(conds, labels, default="NO ID") if conds else "NO ID"

        # Ensure ordering follows GROUPS
        ordered_now = [c for c in ordered_cols if c in df2.columns]
        leftovers_now = [c for c in df2.columns if c not in ordered_cols]
        df2 = df2.reindex(columns=ordered_now + leftovers_now)

        worksheet = workbook.add_worksheet(sheet_name)
        writer.sheets[sheet_name] = worksheet

        worksheet.set_row(0, 20)
        worksheet.set_row(1, 30)

        col_positions = {c: i for i, c in enumerate(df2.columns)}

        # group bands
        for group_name, cols in groups:
            cols_present = [c for c in cols if c in df2.columns]
            if not cols_present:
                continue
            start = col_positions[cols_present[0]]
            end   = col_positions[cols_present[-1]]
            worksheet.merge_range(0, start, 0, end, group_name, group_band_format)

        # headers
        for col_num, col_name in enumerate(df2.columns):
            fmt = header_format if col_name == "Sno" else header_format_match
            worksheet.write(1, col_num, col_name, fmt)

        worksheet.autofilter(1, 0, 1, len(df2.columns) - 1)

        # widths
        for col_num, col in enumerate(df2.columns):
            if col in FIXED_WIDTHS:
                width = FIXED_WIDTHS[col]
            else:
                max_len = max(df2[col].astype(str).map(len).max(), len(col)) + 2
                width = min(max_len, MAX_COL_WIDTH)
                width = max(width, MIN_COL_WIDTH)
            worksheet.set_column(col_num, col_num, width)

        # column indexes
        node_col_index = df2.columns.get_loc("NODE_IDENTIFIER") if "NODE_IDENTIFIER" in df2.columns else None
        tt_col_index   = df2.columns.get_loc("TT_CODES") if "TT_CODES" in df2.columns else None
        ingestion_col_index = df2.columns.get_loc("For Ingestion") if "For Ingestion" in df2.columns else None

        def extract_tt(s: str):
            if not s:
                return None
            m = re.search(r"(tt\d+)", s, flags=re.IGNORECASE)
            return m.group(1).lower() if m else None

        # data rows from row 2
        if "ID" in df2.columns:
            id_col_index = df2.columns.get_loc("ID")
            id_counts = df2["ID"].value_counts(dropna=False).to_dict()

            for r in range(len(df2)):
                row_id = df2.iloc[r]["ID"]
                count = id_counts.get(row_id, 0)
                id_fmt = id_green if count == 1 else id_yellow

                for c in range(len(df2.columns)):
                    value = df2.iloc[r, c]

                    # blanks / NaN / Inf
                    if value is None or (isinstance(value, float) and (math.isnan(value) or math.isinf(value))) or pd.isna(value):
                        worksheet.write(r + 2, c, "", id_fmt if c == id_col_index else border_format)
                        continue

                    # 🔴 Highlight NO ID / NO_ID in For Ingestion
                    if ingestion_col_index is not None and c == ingestion_col_index:
                        vstr = str(value).strip()
                        if vstr in ("NO ID", "NO_ID"):
                            worksheet.write(r + 2, c, vstr, no_id_red_format)
                        else:
                            worksheet.write(r + 2, c, value, border_format)
                        continue

                    # 🔴 NEW: If IDENTIFIERS cell contains "|", highlight in red
                    col_name = df2.columns[c]
                    if col_name in identifier_cols_set and "|" in str(value):
                        worksheet.write(r + 2, c, value, pipe_red_format)
                        continue

                    # NODE_IDENTIFIER hyperlink
                    if node_col_index is not None and c == node_col_index:
                        s = str(value).strip()
                        if s.startswith("entities/"):
                            node_id = s.replace("entities/", "", 1)
                            url = RELTIO_BASE_URL + node_id
                            worksheet.write_url(r + 2, c, url, link_format, node_id)
                        else:
                            worksheet.write(r + 2, c, s, border_format)
                        continue

                    # TT_CODES hyperlink (IMDb)
                    if tt_col_index is not None and c == tt_col_index:
                        s = str(value).strip()
                        tt = extract_tt(s)
                        if tt:
                            url = f"{IMDB_BASE_URL}{tt}/"
                            worksheet.write_url(r + 2, c, url, link_format, tt)
                        else:
                            worksheet.write(r + 2, c, s, border_format)
                        continue

                    # normal write
                    worksheet.write(r + 2, c, value, id_fmt if c == id_col_index else border_format)

        else:
            for r in range(len(df2)):
                for c in range(len(df2.columns)):
                    value = df2.iloc[r, c]
                    if value is None or (isinstance(value, float) and (math.isnan(value) or math.isinf(value))) or pd.isna(value):
                        worksheet.write(r + 2, c, "", border_format)
                        continue

                    # For Ingestion highlight
                    if ingestion_col_index is not None and c == ingestion_col_index:
                        vstr = str(value).strip()
                        worksheet.write(r + 2, c, vstr, no_id_red_format if vstr in ("NO ID", "NO_ID") else border_format)
                        continue

                    # IDENTIFIERS pipe highlight
                    col_name = df2.columns[c]
                    if col_name in identifier_cols_set and "|" in str(value):
                        worksheet.write(r + 2, c, value, pipe_red_format)
                        continue

                    worksheet.write(r + 2, c, value, border_format)

        return df2


    def format_no_match(df, sheet_name):
        worksheet = workbook.add_worksheet(sheet_name)
        writer.sheets[sheet_name] = worksheet

        df_nm = df.copy()
        df_nm = convert_columns_try_number_preserve_decimals(df_nm, NUM_COERCE_COLS)

        for col_num, col_name in enumerate(df_nm.columns):
            worksheet.write(0, col_num, col_name, header_format)

        for col_num, col in enumerate(df_nm.columns):
            max_len = max(df_nm[col].astype(str).map(len).max(), len(col)) + 2
            width = min(max_len, MAX_COL_WIDTH)
            width = max(width, MIN_COL_WIDTH)
            worksheet.set_column(col_num, col_num, width)

        for r in range(len(df_nm)):
            for c in range(len(df_nm.columns)):
                value = df_nm.iloc[r, c]
                if value is None or (isinstance(value, float) and (math.isnan(value) or math.isinf(value))) or pd.isna(value):
                    worksheet.write(r + 1, c, "", border_format)
                else:
                    worksheet.write(r + 1, c, value, border_format)


    # Decide mode (as in your code)
    ip_mode_map = {"S": "STANDALONE", "SEA": "SEASON", "SE": "SERIES", "E": "EPISODICS"}
    ip_mode = ip_mode_map.get(IP, "STANDALONE")

    # SHEET ORDER: Top match -> Partial match -> No match
    apply_grouped_formatting(Matches_top3, "Top match", ip_mode)
    apply_grouped_formatting(Partial_matches, "Partial match", ip_mode)
    format_no_match(No_match_sheet, "No match")

print("Final output is ready...✅")

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://sso.wbd.com/app/snowflake/exkun8q7gmVUtforO0x7/sso/saml?SAMLRequest=lZJdb9owGIX%2FSuRdJ3ZC%2BagFVBmoGhotiK%2BpuzOxk7okdvDrNPDv5wSYuotW2l3knOPn%2BD3v8OFU5N67MCC1GqEwIMgTKtFcqmyEtptHf4A8sExxlmslRugsAD2Mh8CKvKRxZV%2FVShwrAdZzFymgzY8RqoyimoEEqlghgNqEruOnOY0CQhmAMNbh0NXCQTrWq7Ulxbiu66DuBNpkOCKEYHKPnaqRfEMfEOXXjNJoqxOd3ywn96ZPECEmdw3CKRxheTV%2Bl%2Boygq8o%2B4sI6I%2FNZukvF%2BsN8uLb6yZaQVUIsxbmXSZiu5pfAoBL8GsWd3qk1wsq8AUD64cBKF2nOTuIRBdlZd21gfvCqeA415l0w5pNR6g8SF6xrllMdulqe2Dlvl8cuYrrntAv%2FC17Gzyx%2BOczeZkX2ZlvE%2BTtbtVGTbUzgErMVFOodUck6vmk60f3mzCknQHtdgJyN%2FiNvKkrVCpmW%2BctNYAO6j1vc7GyxH8jY3E6VGpw7GfFbmtTbRbk1G%2FkuKkKXbaFtmwz%2Fu8ZDPFH%2B3Xznl0Zs%2BlS5zI5e4%2FaFMx%2B3lUYhO2J5H7aSqkomMxjzo0AcJ3lua4nRjDrFtyaSiA8vlD%2FXfHxHw%3D%3D&RelayState=ver%3A3-hint%3A22419604889210898-ETMsDgAAAZ5zh7D4ABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEIRAoR7Lb4KB0EhYehCm6AoAAACgzoWZu%2BeJ8epoYgYOVn%2BFSL7

import shutil
import os

model_cache = os.path.expanduser("~/.cache/huggingface/hub")

shutil.rmtree(model_cache, ignore_errors=True)

print("Model cache cleared!")